# LLM with TOON Format

This notebook demonstrates using the Typhoon LLM with TOON format for financial data analysis.

We'll:
1. Load financial data from CSV
2. Convert it to TOON format
3. Send it to the LLM for analysis
4. Get the response in TOON format

## Setup

In [1]:
import json
from pandas import read_csv, DataFrame
from toon_format import encode, decode

from pkz.toon.module.prompt import prep_prompts
from pkz.client.llm import get_typhoon_client, generate
from pkz.toon.module.token_counter import TokenCounter
from pkz.toon.module.format_comparator import FormatComparator

In [ ]:
def run_test(
    test_name: str,
    data_input: str,
    data_format: str,
    include_example: bool,
    include_template: bool,
    max_tokens: int = 2048,
) -> dict:
    """
    Run a single test scenario and collect metrics.
    
    Returns:
        Dictionary with test results including tokens and response
    """
    print(f"\n{'='*80}")
    print(f"Running: {test_name}")
    print(f"{'='*80}")
    
    # Prepare prompts
    prompts = prep_prompts(
        data_input=data_input,
        data_foramt=data_format,
        toon_format_example=toon_format_example if include_example else "",
        toon_output_template=toon_output_template if include_template else "",
    )
    
    system_prompt = prompts["system_prompt"]
    user_prompt = prompts["user_prompt"]
    
    # Count input tokens
    input_tokens = counter.count(system_prompt) + counter.count(user_prompt)
    
    print(f"\nInput tokens: {input_tokens}")
    print(f"  System prompt: {counter.count(system_prompt)}")
    print(f"  User prompt: {counter.count(user_prompt)}")
    
    # Generate response
    print("\nCalling LLM...")
    response = generate(
        client=client,
        user_prompt=user_prompt,
        system_prompt=system_prompt,
        max_tokens=max_tokens,
        temperature=0.1,
    )
    
    # Count output tokens
    output_tokens = counter.count(response)
    total_tokens = input_tokens + output_tokens
    
    print(f"\nOutput tokens: {output_tokens}")
    print(f"Total tokens: {total_tokens}")
    
    # Preview response
    print(f"\nResponse preview:")
    print("-" * 80)
    print(response)
    print("-" * 80)
    
    return {
        "test_name": test_name,
        "data_format": data_format,
        "include_example": include_example,
        "include_template": include_template,
        "system_prompt_tokens": counter.count(system_prompt),
        "user_prompt_tokens": counter.count(user_prompt),
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": total_tokens,
        "response": response,
        "system_prompt": system_prompt,
        "user_prompt": user_prompt,
    }

def run_traditional_test(
    test_name: str,
    data_input: str,
    input_format: str,
    output_format: str,
    compact: bool = False,
    max_tokens: int = 2048,
) -> dict:
    """
    Run a traditional format test (e.g., CSV input with JSON output).
    
    Returns:
        Dictionary with test results including tokens and response
    """
    print(f"\n{'='*80}")
    print(f"Running: {test_name}")
    print(f"{'='*80}")
    
    # Create custom prompts for traditional format
    system_prompt = f"""You are a financial analyst expert. You will receive financial data in {input_format} format and must provide analysis in {output_format} format."""
    
    output_instructions = ""
    if output_format == "JSON":
        if compact:
            output_instructions = "Return your response as a single compact JSON object (no extra whitespace) with three keys: key_findings, trend_analysis, and health_assessment. Each should be an array of objects."
        else:
            output_instructions = "Return your response as a pretty-printed JSON object with three keys: key_findings, trend_analysis, and health_assessment. Each should be an array of objects."
    
    user_prompt = f"""Analyze the following financial data for a company over multiple years (Thai fiscal years: 2565=2022, 2566=2023, 2567=2024, 2568=2025).

Financial Data ({input_format} format):
{data_input}

Please provide:
1. A summary of key findings (with fields: finding, description, impact)
2. Trend analysis for key metrics (with fields: metric, trend, change_pct, assessment)
3. Financial health assessment (with fields: category, score, comment)

{output_instructions}"""
    
    # Count input tokens
    input_tokens = counter.count(system_prompt) + counter.count(user_prompt)
    
    print(f"\nInput tokens: {input_tokens}")
    print(f"  System prompt: {counter.count(system_prompt)}")
    print(f"  User prompt: {counter.count(user_prompt)}")
    
    # Generate response
    print("\nCalling LLM...")
    response = generate(
        client=client,
        user_prompt=user_prompt,
        system_prompt=system_prompt,
        max_tokens=max_tokens,
        temperature=0.1,
    )
    
    # Count output tokens
    output_tokens = counter.count(response)
    total_tokens = input_tokens + output_tokens
    
    print(f"\nOutput tokens: {output_tokens}")
    print(f"Total tokens: {total_tokens}")
    
    # Preview response
    print(f"\nResponse preview (first 500 chars):")
    print("-" * 80)
    print(response[:500])
    print("-" * 80)
    
    return {
        "test_name": test_name,
        "data_format": f"{input_format} → {output_format}" + (" (compact)" if compact else ""),
        "include_example": False,
        "include_template": False,
        "system_prompt_tokens": counter.count(system_prompt),
        "user_prompt_tokens": counter.count(user_prompt),
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": total_tokens,
        "response": response,
        "system_prompt": system_prompt,
        "user_prompt": user_prompt,
    }

# Simple text-based bar chart for token comparison
def text_bar_chart(values, labels, max_width=60):
    """Create a simple text-based bar chart."""
    max_val = max(values)
    
    for val, label in zip(values, labels):
        bar_width = int((val / max_val) * max_width)
        bar = "█" * bar_width
        print(f"{label:40} {bar} {val:,}")


## Load Financial Data

In [3]:
# Load CSV data
df = read_csv("data/example1.csv")
df

,รายการเงินสด วันที่,งบปี 2565 (01 ม.ค. 2565 - 31 ธ.ค. 2565),งบปี 2566 (01 ม.ค. 2566 - 31 ธ.ค. 2566),งบปี 2567 (01 ม.ค. 2567 - 31 ธ.ค. 2567),งบ 9 เดือน 2568 (01 ม.ค. 2568 - 30 ก.ย. 2568)
0,สินทรัพย์รวม,3454452.34,3438721.93,3486539.22,3533711.71
1,หนี้สินรวม,2987840.45,2954988.62,2991702.15,3041296.16
2,ส่วนของผู้ถือหุ้น,461231.65,478082.22,488635.60,486162.95
3,มูลค่าหุ้นที่เรียกชำระแล้ว,33671.07,33671.07,33671.07,33671.07
4,รายได้รวม,177547.04,210556.79,215706.27,153811.55
5,กำไร (ขาดทุน) จากกิจกรรมอื่น,6996.20,9313.49,9239.62,13452.72
6,กำไรสุทธิ,37546.01,43521.33,43943.01,37344.41
7,กำไรต่อหุ้น (บาท),11.12,12.93,13.05,11.09
8,ROA (%),2.02,2.67,2.79,2.91
9,ROE (%),8.14,9.27,9.09,10.18


In [4]:
# Convert to dict format for TOON encoding
data = df.to_dict(orient="records")
print(data)
print(f"Loaded {len(data)} financial metrics")

[{'รายการเงินสด วันที่': 'สินทรัพย์รวม', 'งบปี 2565 (01 ม.ค. 2565 - 31 ธ.ค. 2565)': 3454452.34, 'งบปี 2566 (01 ม.ค. 2566 - 31 ธ.ค. 2566)': 3438721.93, 'งบปี 2567 (01 ม.ค. 2567 - 31 ธ.ค. 2567)': 3486539.22, 'งบ 9 เดือน 2568 (01 ม.ค. 2568 - 30 ก.ย. 2568)': 3533711.71}, {'รายการเงินสด วันที่': 'หนี้สินรวม', 'งบปี 2565 (01 ม.ค. 2565 - 31 ธ.ค. 2565)': 2987840.45, 'งบปี 2566 (01 ม.ค. 2566 - 31 ธ.ค. 2566)': 2954988.62, 'งบปี 2567 (01 ม.ค. 2567 - 31 ธ.ค. 2567)': 2991702.15, 'งบ 9 เดือน 2568 (01 ม.ค. 2568 - 30 ก.ย. 2568)': 3041296.16}, {'รายการเงินสด วันที่': 'ส่วนของผู้ถือหุ้น', 'งบปี 2565 (01 ม.ค. 2565 - 31 ธ.ค. 2565)': 461231.65, 'งบปี 2566 (01 ม.ค. 2566 - 31 ธ.ค. 2566)': 478082.22, 'งบปี 2567 (01 ม.ค. 2567 - 31 ธ.ค. 2567)': 488635.6, 'งบ 9 เดือน 2568 (01 ม.ค. 2568 - 30 ก.ย. 2568)': 486162.95}, {'รายการเงินสด วันที่': 'มูลค่าหุ้นที่เรียกชำระแล้ว', 'งบปี 2565 (01 ม.ค. 2565 - 31 ธ.ค. 2565)': 33671.07, 'งบปี 2566 (01 ม.ค. 2566 - 31 ธ.ค. 2566)': 33671.07, 'งบปี 2567 (01 ม.ค. 2567 - 31 ธ.ค. 2567)

## Convert to TOON Format

In [5]:
# Convert data to TOON format
data_toon = encode(data, options={"delimiter": "\t"})

print("Financial Data in TOON Format:")
print("=" * 80)
print(data_toon)
print("=" * 80)

Financial Data in TOON Format:
[11	]{"รายการเงินสด วันที่"	"งบปี 2565 (01 ม.ค. 2565 - 31 ธ.ค. 2565)"	"งบปี 2566 (01 ม.ค. 2566 - 31 ธ.ค. 2566)"	"งบปี 2567 (01 ม.ค. 2567 - 31 ธ.ค. 2567)"	"งบ 9 เดือน 2568 (01 ม.ค. 2568 - 30 ก.ย. 2568)"}:
  สินทรัพย์รวม	3454452.34	3438721.93	3486539.22	3533711.71
  หนี้สินรวม	2987840.45	2954988.62	2991702.15	3041296.16
  ส่วนของผู้ถือหุ้น	461231.65	478082.22	488635.6	486162.95
  มูลค่าหุ้นที่เรียกชำระแล้ว	33671.07	33671.07	33671.07	33671.07
  รายได้รวม	177547.04	210556.79	215706.27	153811.55
  กำไร (ขาดทุน) จากกิจกรรมอื่น	6996.2	9313.49	9239.62	13452.72
  กำไรสุทธิ	37546.01	43521.33	43943.01	37344.41
  กำไรต่อหุ้น (บาท)	11.12	12.93	13.05	11.09
  ROA (%)	2.02	2.67	2.79	2.91
  ROE (%)	8.14	9.27	9.09	10.18
  อัตรากำไรสุทธิ (%)	20.91	20.79	20.67	24.69


## LLM Analysis with TOON Format

We'll ask the LLM to analyze the financial data and return insights in TOON format.

In [6]:
# Prepare supplementary information
toon_format_example = """
TOON format example for tabular data:
[2\t]{metric\tvalue\ttrend}:
  ROA\t2.5\tincreasing
  ROE\t9.2\tstable

"""

toon_output_template = """

Use the following structure (N, M, K are the number of rows):
key_findings[N\t]{{finding\tdescription\timpact}}:
  ...
trend_analysis[M\t]{{metric\ttrend\tchange_pct\tassessment}}:
  ...
health_assessment[K\t]{{category\tscore\tcomment}}:
  ...
"""

# Initialize client and token counter
client = get_typhoon_client()
counter = TokenCounter()
comparator = FormatComparator()

## Systematic Testing Framework

We'll test 5 scenarios to evaluate:
1. **LLM's ability to recognize TOON format** without examples
2. **Impact of providing examples and templates** on output quality and token usage
3. **Token efficiency** of TOON format vs traditional approaches

### Test Scenarios:
1. No examples, no template (bare minimum)
2. With `toon_format_example` only
3. With `toon_output_template` only
4. With both `toon_format_example` and `toon_output_template`
5. Traditional: CSV input → JSON output

In [7]:
# Prepare different input formats for testing
data_csv = comparator.display_csv(data)
data_json = comparator.display_json(data)
data_json_compact = comparator.display_json_compact(data)

print("CSV format preview:")
print(data_csv[:300] + "...\n")

print(f"Token counts for input formats:")
print(f"  TOON: {counter.count(data_toon)} tokens")
print(f"  CSV: {counter.count(data_csv)} tokens")
print(f"  JSON: {counter.count(data_json)} tokens")
print(f"  JSON compact: {counter.count(data_json_compact)} tokens")

CSV format preview:
รายการเงินสด วันที่,งบปี 2565 (01 ม.ค. 2565 - 31 ธ.ค. 2565),งบปี 2566 (01 ม.ค. 2566 - 31 ธ.ค. 2566),งบปี 2567 (01 ม.ค. 2567 - 31 ธ.ค. 2567),งบ 9 เดือน 2568 (01 ม.ค. 2568 - 30 ก.ย. 2568)
สินทรัพย์รวม,3454452.34,3438721.93,3486539.22,3533711.71
...้สินรวม,2987840.45,2954988.62,2991702.15,3041296.16

Token counts for input formats:
  TOON: 420 tokens
  CSV: 390 tokens
  JSON: 4414 tokens
  JSON compact: 4226 tokens


## Run All Tests

In [8]:
# Initialize results storage
results = []

# Test 1: TOON format - No examples, no template
results.append(run_test(
    test_name="Test 1: TOON (no example, no template)",
    data_input=data_toon,
    data_format="TOON",
    include_example=False,
    include_template=False,
))


Running: Test 1: TOON (no example, no template)

Input tokens: 603
  System prompt: 38
  User prompt: 565

Calling LLM...

Output tokens: 750
Total tokens: 1353

Response preview (first 500 chars):
--------------------------------------------------------------------------------
```json
{
  "key_findings": [
    {
      "finding": "Revenue Growth Slowdown",
      "description": "Revenue growth decelerated significantly in 2568 (9 months) compared to previous years.",
      "impact": "Potential concerns about market share or competitive pressures. Requires further investigation into sales performance and market conditions."
    },
    {
      "finding": "Profitability Remains Strong",
      "description": "Despite revenue slowdown, net profit remained relatively stable i
--------------------------------------------------------------------------------


In [9]:
# Test 2: TOON format - With `toon_format_example` only
results.append(run_test(
    test_name="Test 2: TOON (example only)",
    data_input=data_toon,
    data_format="TOON",
    include_example=True,
    include_template=False,
))


Running: Test 2: TOON (example only)

Input tokens: 642
  System prompt: 77
  User prompt: 565

Calling LLM...

Output tokens: 562
Total tokens: 1204

Response preview (first 500 chars):
--------------------------------------------------------------------------------
```toon
[key_findings]{"finding":"Revenue Growth Slowdown","description":"Revenue growth slowed significantly in 2568 (9 months) compared to previous years.","impact":"Potential concerns about market share or competitive pressures."}
[key_findings]{"finding":"Profitability Remains Strong","description":"Despite revenue slowdown, net profit remains robust and even increased slightly in 2568 (9 months).","impact":"Indicates effective cost management or pricing strategies."}
[key_findings]{"finding
--------------------------------------------------------------------------------


In [10]:
# Test 3: TOON format - With `toon_output_template` only
results.append(run_test(
    test_name="Test 3: TOON (template only)",
    data_input=data_toon,
    data_format="TOON",
    include_example=False,
    include_template=True,
))


Running: Test 3: TOON (template only)

Input tokens: 666
  System prompt: 38
  User prompt: 628

Calling LLM...

Output tokens: 657
Total tokens: 1323

Response preview (first 500 chars):
--------------------------------------------------------------------------------
```json
{
  "key_findings": [
    {
      "finding": "Revenue Growth Slowdown",
      "description": "Revenue growth slowed significantly in 2568 (Jan-Sep) compared to previous years.",
      "impact": "Potential concerns about market demand or competitive pressures."
    },
    {
      "finding": "Profitability Remains Strong",
      "description": "Despite revenue slowdown, net profit remains robust, indicating efficient cost management.",
      "impact": "Positive sign for long-term sustainab
--------------------------------------------------------------------------------


In [11]:
# Test 4: TOON format - With both `toon_format_example` and `toon_output_template`
results.append(run_test(
    test_name="Test 4: TOON (both)",
    data_input=data_toon,
    data_format="TOON",
    include_example=True,
    include_template=True,
))


Running: Test 4: TOON (both)

Input tokens: 705
  System prompt: 77
  User prompt: 628

Calling LLM...

Output tokens: 502
Total tokens: 1207

Response preview (first 500 chars):
--------------------------------------------------------------------------------
```toon
key_findings[4	]{{finding	description	impact}}:
  Revenue Growth Slowdown	Revenue growth slowed significantly in 2568 (Jan-Sep) compared to previous years.	Potential impact on future profitability and expansion plans.
  Profitability Stability	Net profit remained relatively stable between 2566 and 2567 but decreased in 2568 (Jan-Sep).	Requires investigation into cost management and revenue diversification.
  ROE Improvement	ROE continued to improve, indicating efficient use of shareholde
--------------------------------------------------------------------------------


In [12]:
# Test 5: Traditional format - CSV input with JSON output
results.append(run_traditional_test(
    test_name="Test 5: CSV → JSON (pretty-printed)",
    data_input=data_csv,
    input_format="CSV",
    output_format="JSON",
    compact=False,
))


Running: Test 5: CSV → JSON (pretty-printed)

Input tokens: 553
  System prompt: 23
  User prompt: 530

Calling LLM...

Output tokens: 768
Total tokens: 1321

Response preview (first 500 chars):
--------------------------------------------------------------------------------
```json
{
  "key_findings": [
    {
      "finding": "Revenue Growth",
      "description": "Revenue increased from 2565 to 2566, then slightly decreased in 2567 and significantly dropped in the first 9 months of 2568.",
      "impact": "The decline in revenue during 2568 could indicate market challenges or internal issues that need addressing."
    },
    {
      "finding": "Profitability",
      "description": "Net profit consistently remained positive across all periods, with a slight increas
--------------------------------------------------------------------------------


In [34]:
# Input tokens comparison
print("\nINPUT TOKENS (System + User Prompts)")
print("-" * 100)
input_tokens = [r["input_tokens"] for r in results]
labels = [r["test_name"] for r in results]
text_bar_chart(input_tokens, labels)

# Output tokens comparison
print("\n\nOUTPUT TOKENS (LLM Responses)")
print("-" * 100)
output_tokens = [r["output_tokens"] for r in results]
text_bar_chart(output_tokens, labels)

# Total tokens comparison
print("\n\nTOTAL TOKENS (Input + Output)")
print("-" * 100)
total_tokens = [r["total_tokens"] for r in results]
text_bar_chart(total_tokens, labels)


INPUT TOKENS (System + User Prompts)
----------------------------------------------------------------------------------------------------
Test 1: TOON (no example, no template)   ███████████████████████████████████████████████████ 603
Test 2: TOON (example only)              ██████████████████████████████████████████████████████ 642
Test 3: TOON (template only)             ████████████████████████████████████████████████████████ 666
Test 4: TOON (both)                      ████████████████████████████████████████████████████████████ 705
Test 5: CSV → JSON (pretty-printed)      ███████████████████████████████████████████████ 553


OUTPUT TOKENS (LLM Responses)
----------------------------------------------------------------------------------------------------
Test 1: TOON (no example, no template)   ██████████████████████████████████████████████████████████ 750
Test 2: TOON (example only)              ███████████████████████████████████████████ 562
Test 3: TOON (template only)         

## Summary & Conclusions

Based on the systematic testing, we can answer the research questions:

In [15]:
# Display full responses for manual inspection
print("=" * 100)
print("FULL RESPONSES")
print("=" * 100)

for i, result in enumerate(results, 1):
    print(f"\n{'='*100}")
    print(f"{result['test_name']}")
    print(f"{'='*100}")
    print(result['response'])
    print()

FULL RESPONSES

Test 1: TOON (no example, no template)
```json
{
  "key_findings": [
    {
      "finding": "Revenue Growth Slowdown",
      "description": "Revenue growth decelerated significantly in 2568 (9 months) compared to previous years.",
      "impact": "Potential concerns about market share or competitive pressures. Requires further investigation into sales performance and market conditions."
    },
    {
      "finding": "Profitability Remains Strong",
      "description": "Despite revenue slowdown, net profit remained relatively stable in 2568 (9 months), indicating efficient cost management.",
      "impact": "Positive sign for operational efficiency. However, sustained profitability depends on addressing revenue challenges."
    },
    {
      "finding": "Improved ROE",
      "description": "ROE consistently improved over the period, reaching a peak in 2568 (9 months).",
      "impact": "Indicates effective utilization of shareholder equity to generate profits. A positive

In [29]:
# Function to check if response is valid TOON format
def is_valid_toon_format(response: str) -> bool:
    """
    Check if the response appears to be valid TOON format.
    """
    response = (
        response
        .replace("```", "")
        .replace("toon", "")
        .replace("json", "")
        .replace("{{", "{")
        .replace("}}", "}")
    )
    try: 
        decode(response)
        return True
    except:
        return False

# Analyze each TOON test response
print("=" * 100)
print("RESPONSE FORMAT VALIDATION")
print("=" * 100)

for i, result in enumerate(results[:4], 1):
    test_name = result["test_name"]
    response = result["response"]
    is_valid = is_valid_toon_format(response)
    
    print(f"\n{test_name}")
    print(f"  Valid TOON format: {'✓ YES' if is_valid else '✗ NO'}")
    print(f"  Response length: {len(response)} chars")
    
    # Try to decode if it looks like TOON
    if is_valid:
        print(f"  ✓ Response appears to follow TOON format structure")
    else:
        print(f"  ✗ Response does not follow TOON format structure")

print("\n" + "=" * 100)

RESPONSE FORMAT VALIDATION

Test 1: TOON (no example, no template)
  Valid TOON format: ✗ NO
  Response length: 3346 chars
  ✗ Response does not follow TOON format structure

Test 2: TOON (example only)
  Valid TOON format: ✗ NO
  Response length: 2660 chars
  ✗ Response does not follow TOON format structure

Test 3: TOON (template only)
  Valid TOON format: ✗ NO
  Response length: 2873 chars
  ✗ Response does not follow TOON format structure

Test 4: TOON (both)
  Valid TOON format: ✓ YES
  Response length: 1997 chars
  ✓ Response appears to follow TOON format structure



## Response Quality Analysis

Examine the LLM responses to assess whether it correctly understood and generated TOON format.

In [30]:
# Create comparison DataFrame
comparison_df = DataFrame([
    {
        "Test": r["test_name"],
        "Format": r["data_format"],
        "Has Example": "✓" if r["include_example"] else "✗",
        "Has Template": "✓" if r["include_template"] else "✗",
        "Input Tokens": r["input_tokens"],
        "Output Tokens": r["output_tokens"],
        "Total Tokens": r["total_tokens"],
    }
    for r in results
])

print("=" * 100)
print("TOKEN USAGE COMPARISON")
print("=" * 100)
print(comparison_df.to_string(index=False))
print("=" * 100)

TOKEN USAGE COMPARISON
                                  Test     Format Has Example Has Template  Input Tokens  Output Tokens  Total Tokens
Test 1: TOON (no example, no template)       TOON           ✗            ✗           603            750          1353
           Test 2: TOON (example only)       TOON           ✓            ✗           642            562          1204
          Test 3: TOON (template only)       TOON           ✗            ✓           666            657          1323
                   Test 4: TOON (both)       TOON           ✓            ✓           705            502          1207
   Test 5: CSV → JSON (pretty-printed) CSV → JSON           ✗            ✗           553            768          1321
